In [ ]:
# Install dependencies (clean & safe)

!pip install -q segmentation-models-pytorch==0.2.1 timm==0.4.12 albumentations==1.3.1 opencv-python-headless scikit-image

print("✅ Packages installed. Restart runtime manually if needed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 16.4 MB/s eta 0:00:00
✅ Packages installed. Restart runtime manually if needed.


In [ ]:
# CELL 0 — Clean reset + inspect FALLMuD.zip

import os, zipfile

DATA_ZIP = "/content/FALLMUD.zip"
EXTRACT_DIR = "/content/FALLMUD"

# Hard reset (safe)
os.system(f"rm -rf {EXTRACT_DIR}")

# Confirm dataset exists
print("FALLMUD.zip exists:", os.path.exists(DATA_ZIP))

# Inspect ZIP structure
with zipfile.ZipFile(DATA_ZIP, "r") as zf:
    names = zf.namelist()
    print("\nFirst 30 entries inside FALLMUD.zip:")
    for name in names[:30]:
        print(" ", name)

# Extract dataset
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("\n✅ Extracted to:", EXTRACT_DIR)
print("Top-level folders in FALLMUD:")
print(os.listdir(EXTRACT_DIR))


FALLMUD.zip exists: True

First 30 entries inside FALLMUD.zip:
  FALLMUD/ReadMe.txt
  FALLMUD/.DS_Store
  FALLMUD/NeilCronin/.DS_Store
  FALLMUD/RyanCunningham/.DS_Store
  FALLMUD/.venv/pyvenv.cfg
  FALLMUD/.venv/.gitignore
  FALLMUD/.venv/bin/activate.fish
  FALLMUD/.venv/bin/activate.csh
  FALLMUD/.venv/bin/python3.9
  FALLMUD/.venv/bin/Activate.ps1
  FALLMUD/.venv/bin/pip3.9
  FALLMUD/.venv/include/
  FALLMUD/.venv/lib/python3.9-site-packages-pip-_vendor-pygments-LICENSE
  FALLMUD/.venv/lib/python3.9-site-packages-pip-_vendor-msgpack-COPYING
  FALLMUD/.venv/lib/python3.9-site-packages-pip-_vendor-requests-LICENSE
  FALLMUD/.venv/lib/python3.9-site-packages-pip-_vendor-distro-LICENSE
  FALLMUD/.venv/lib/python3.9-site-packages-pip-_vendor-packaging-LICENSE
  FALLMUD/.venv/lib/python3.9-site-packages-pip-_vendor-truststore-LICENSE
  FALLMUD/.venv/lib/python3.9-site-packages-pip-_vendor-pyproject_hooks-LICENSE
  FALLMUD/.venv/lib/python3.9-site-packages-pip-_vendor-certifi-LICENSE
  FA

In [ ]:
# CELL 1 — Clean RGB dataset setup (NO preprocessing, NO geometry edits)

import os, shutil, cv2
from pathlib import Path
from tqdm import tqdm

DATA_ROOT = Path("/content/FALLMUD/FALLMUD")
OUT_DIR = Path("/content/RGB_READY")   # clean working copy
OUT_DIR.mkdir(exist_ok=True)

def prepare_dataset(split):
    src_img = DATA_ROOT / split / "images"
    src_msk = DATA_ROOT / split / "RGB_mask"

    dst_img = OUT_DIR / split / "images"
    dst_msk = OUT_DIR / split / "RGB_mask"

    dst_img.mkdir(parents=True, exist_ok=True)
    dst_msk.mkdir(parents=True, exist_ok=True)

    files = sorted(os.listdir(src_img))

    print(f"\nCopying {split} data...")

    for f in tqdm(files):
        stem = Path(f).stem
        img_path = src_img / f
        msk_path = src_msk / f"{stem}_combined.png"

        if not msk_path.exists():
            print("❌ Missing mask:", msk_path.name)
            continue

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        msk = cv2.imread(str(msk_path), cv2.IMREAD_COLOR)

        # sanity checks
        if img is None or msk is None:
            print("❌ Corrupt file:", f)
            continue

        if img.shape[:2] != msk.shape[:2]:
            print("❌ Size mismatch:", f, img.shape, msk.shape)
            continue

        shutil.copy(img_path, dst_img / f)
        shutil.copy(msk_path, dst_msk / msk_path.name)

    print(f"✅ {split} ready.")


for sub in ["NeilCronin", "RyanCunningham"]:
    prepare_dataset(sub)

print("\n✅ DATASET READY IN:", OUT_DIR)




Copying NeilCronin data...


100%|██████████| 309/309 [00:02<00:00, 142.53it/s]


✅ NeilCronin ready.

Copying RyanCunningham data...


100%|██████████| 504/504 [00:01<00:00, 252.43it/s]

✅ RyanCunningham ready.

✅ DATASET READY IN: /content/RGB_READY


In [ ]:
# ==== QC CHECK (NEW RGB DATASET) — Verify RGB masks & sizes ====

import cv2, numpy as np
from pathlib import Path
from tqdm import tqdm

BASE = Path("/content/RGB_READY/RyanCunningham")
IMG_DIR = BASE / "images"
MSK_DIR = BASE / "RGB_mask"

assert IMG_DIR.exists(), f"Images not found: {IMG_DIR}"
assert MSK_DIR.exists(), f"Masks not found: {MSK_DIR}"

mismatch = 0
cnt_red = cnt_green = cnt_yellow = 0
missing_img = 0
missing_mask = 0
total = 0

for img_path in tqdm(sorted(IMG_DIR.iterdir())):
    if img_path.suffix.lower() not in [".jpg", ".png", ".tif"]:
        continue

    stem = img_path.stem
    mask_path = MSK_DIR / f"{stem}_combined.png"

    # read image
    im = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if im is None:
        missing_img += 1
        continue

    # read mask
    m_bgr = cv2.imread(str(mask_path), cv2.IMREAD_COLOR)
    if m_bgr is None:
        missing_mask += 1
        continue

    m = cv2.cvtColor(m_bgr, cv2.COLOR_BGR2RGB)
    total += 1

    # size check
    if im.shape[:2] != m.shape[:2]:
        mismatch += 1

    # color presence
    red_mask    = (m[:,:,0] == 255) & (m[:,:,1] == 0)   & (m[:,:,2] == 0)
    green_mask  = (m[:,:,0] == 0)   & (m[:,:,1] == 255) & (m[:,:,2] == 0)
    yellow_mask = (m[:,:,0] == 255) & (m[:,:,1] == 255) & (m[:,:,2] == 0)

    cnt_red    += int(np.any(red_mask))
    cnt_green  += int(np.any(green_mask))
    cnt_yellow += int(np.any(yellow_mask))


print("\n==== QC SUMMARY ====")
print(f"Total checked : {total}")
print(f"Size mismatch : {mismatch}")
print(f"Has SUP (red) : {cnt_red}")
print(f"Has FASC (green) : {cnt_green}")
print(f"Has DEEP (yellow) : {cnt_yellow}")
print(f"Missing images : {missing_img}")
print(f"Missing masks  : {missing_mask}")


100%|██████████| 504/504 [00:02<00:00, 168.45it/s]


==== QC SUMMARY ====
Total checked : 504
Size mismatch : 0
Has SUP (red) : 504
Has FASC (green) : 503
Has DEEP (yellow) : 504
Missing images : 0
Missing masks  : 0


In [ ]:
# ==== CELL 2 (FINAL): Dataset + Model + Loss (SUP/DEEP only) ====

import cv2, albumentations as A
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np, pandas as pd, hashlib, random
from pathlib import Path
import segmentation_models_pytorch as smp

# -------------------- Reproducibility --------------------
SEED, IMG_SIZE, BATCH_SIZE = 1337, 640, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -------------------- Load dataset --------------------
BASE = Path("/content/RGB_READY/RyanCunningham")
IMG_DIR = BASE / "images"
MSK_DIR = BASE / "RGB_mask"

rows = []
for ip in sorted(IMG_DIR.iterdir()):
    if ip.suffix.lower() not in [".jpg",".png",".tif"]:
        continue
    mp = MSK_DIR / f"{ip.stem}_combined.png"
    if mp.exists():
        rows.append({"image": str(ip), "mask_rgb": str(mp)})

df = pd.DataFrame(rows)
print("✅ Loaded dataset:", len(df), "pairs")

# -------------------- Stable 80/10/10 split --------------------
SPLIT_DIR = Path("/content/splits"); SPLIT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATHS = {
    "train": SPLIT_DIR/"train.csv",
    "val":   SPLIT_DIR/"val.csv",
    "test":  SPLIT_DIR/"test.csv",
}

def stable_split(frame, seed=SEED):
    keys = (frame["image"].astype(str) + f"|{seed}").apply(
        lambda s: int(hashlib.md5(s.encode()).hexdigest(), 16)
    )
    idx = np.argsort(keys.values)
    f = frame.iloc[idx].reset_index(drop=True)
    n = len(f); n_tr = int(0.8*n); n_va = int(0.1*n)
    parts = {"train": f[:n_tr], "val": f[n_tr:n_tr+n_va], "test": f[n_tr+n_va:]}
    for k,v in parts.items():
        v.to_csv(SPLIT_PATHS[k], index=False)
    return parts

if all(p.exists() for p in SPLIT_PATHS.values()):
    train_df, val_df, test_df = (pd.read_csv(SPLIT_PATHS["train"]),
                                 pd.read_csv(SPLIT_PATHS["val"]),
                                 pd.read_csv(SPLIT_PATHS["test"]))
else:
    parts = stable_split(df)
    train_df, val_df, test_df = parts["train"], parts["val"], parts["test"]

print(f"✅ Split: {len(train_df)} train / {len(val_df)} val / {len(test_df)} test")


# -------------------- RGB → INDEX (SUP/DEEP ONLY) --------------------
def rgb_to_index(mask_rgb):
    """0=BG, 1=SUP, 2=DEEP (fascicle ignored → BG)"""
    m = np.zeros(mask_rgb.shape[:2], np.uint8)
    R,G,B = mask_rgb[:,:,0], mask_rgb[:,:,1], mask_rgb[:,:,2]
    m[(R==255) & (G==0)   & (B==0)]   = 1  # Superficial
    m[(R==255) & (G==255) & (B==0)]   = 2  # Deep
    return m


# -------------------- Augmentations (no anatomy distortion) --------------------
train_tf = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=1.0),
    A.Sharpen(p=1.0),
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.RandomBrightnessContrast(p=0.2),
    A.GaussianBlur(blur_limit=3, p=0.1),
    A.GaussNoise(var_limit=(10,30), p=0.15),
], is_check_shapes=False)

val_tf = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=1.0),
    A.Sharpen(p=1.0),
    A.Resize(IMG_SIZE, IMG_SIZE),
], is_check_shapes=False)


# -------------------- Dataset --------------------
class SRBDDataset(Dataset):
    def __init__(self, frame, tf):
        self.items = frame[["image","mask_rgb"]].to_numpy()
        self.tf = tf

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        ip, mp = self.items[i]
        img = cv2.imread(ip, cv2.IMREAD_GRAYSCALE)
        rgb = cv2.cvtColor(cv2.imread(mp), cv2.COLOR_BGR2RGB)
        if img is None or rgb is None:
            raise FileNotFoundError(ip, mp)

        aug = self.tf(image=img, mask=rgb)
        y = rgb_to_index(aug["mask"])
        x = torch.from_numpy(aug["image"][None].astype(np.float32) / 255.0)
        y = torch.from_numpy(y.astype(np.int64))
        return x, y


tr_ds, va_ds = SRBDDataset(train_df, train_tf), SRBDDataset(val_df, val_tf)


# -------------------- Oversample images that actually contain aponeuroses --------------------
def compute_weights(frame):
    ws=[]
    for _, r in frame.iterrows():
        m = rgb_to_index(cv2.cvtColor(cv2.imread(r.mask_rgb), cv2.COLOR_BGR2RGB))
        ws.append(4.0 if ((m==1)|(m==2)).sum() > 500 else 1.0)
    return torch.DoubleTensor(ws)

sampler = WeightedRandomSampler(compute_weights(train_df), num_samples=len(train_df), replacement=True)

dl_tr  = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
dl_val = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ DataLoaders ready | Train batches: {len(dl_tr)} | Val batches: {len(dl_val)}")


# -------------------- Model (3 classes: BG/SUP/DEEP) --------------------
model = smp.Unet("timm-mobilenetv3_large_100", in_channels=1, classes=3).to(DEVICE)
# If OOM, fall back to:
# model = smp.Unet("resnet34", in_channels=1, classes=3).to(DEVICE)


# -------------------- Loss (force SUP/DEEP + boundary) --------------------
ce_loss = nn.CrossEntropyLoss(weight=torch.tensor([0.05, 6.0, 6.0], device=DEVICE))

LAPL = torch.tensor([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=torch.float32, device=DEVICE).view(1,1,3,3)

def boundary_loss(logits, y):
    p  = torch.softmax(logits, dim=1)
    y1 = F.one_hot(y, 3).permute(0,3,1,2).float()

    def lap(t):
        return F.conv2d(
            t, LAPL.expand(t.size(1),1,3,3),
            padding=1, groups=t.size(1)
        ).abs()

    # Only SUP/DEEP edges
    return F.l1_loss(lap(p[:,1:]), lap(y1[:,1:]))

def total_loss(logits, y):
    return ce_loss(logits, y) + 0.35 * boundary_loss(logits, y)

print("✅ Model and loss ready (SUP/DEEP forced)")





✅ Loaded dataset: 504 pairs
✅ Split: 403 train / 50 val / 51 test


/usr/local/lib/python3.12/dist-packages/albumentations/augmentations/blur/transforms.py:184: UserWarning: blur_limit and sigma_limit minimum value can not be both equal to 0. blur_limit minimum value changed to 3.
  warnings.warn(


✅ DataLoaders ready | Train batches: 68 | Val batches: 9
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_mobilenetv3_large_100-427764d5.pth" to /root/.cache/torch/hub/checkpoints/tf_mobilenetv3_large_100-427764d5.pth


100%|██████████| 21.1M/21.1M [00:01<00:00, 14.6MB/s]


✅ Model and loss ready (SUP/DEEP forced)


In [ ]:
# ==== CELL 3: Training + Dice (SUP/DEEP only) + Best Model Save ====

from tqdm import tqdm
import os, numpy as np, torch

# ---- paths ----
os.makedirs("/content/srbd_training_out", exist_ok=True)
best_path = "/content/srbd_training_out/best_model.pt"

# ---- training params ----
EPOCHS = 40
LR = 2e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

AMP = True
scaler = torch.cuda.amp.GradScaler(enabled=AMP)

best_mean = 0.0


# ---------- Dice for SUP/DEEP only ----------
def dice_sup_deep(pred_logits, target):
    pred = torch.argmax(pred_logits, dim=1)
    dices = []
    for c in [1, 2]:   # SUP, DEEP
        p = (pred == c).float()
        t = (target == c).float()
        inter = (p * t).sum()
        union = p.sum() + t.sum()
        dices.append((2 * inter / (union + 1e-6)).item())
    return dices   # [SUP, DEEP]


# ---------- Training Loop ----------
for epoch in range(1, EPOCHS + 1):

    # ---- TRAIN ----
    model.train()
    train_loss = 0.0

    for x, y in tqdm(dl_tr, desc=f"[Train {epoch}]"):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=AMP):
            logits = model(x)
            loss = total_loss(logits, y)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()


    # ---- VALIDATION (Dice SUP/DEEP) ----
    model.eval()
    dices = []

    with torch.no_grad():
        for x, y in dl_val:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            out = model(x)
            d = dice_sup_deep(out, y)
            dices.append(d)

    sup, deep = np.mean(dices, axis=0)
    mean_sd = (sup + deep) / 2

    print(f"[Epoch {epoch:02d}] "
          f"loss={train_loss/len(dl_tr):.4f} | "
          f"SUP={sup:.3f}  DEEP={deep:.3f}  MEAN(SD)={mean_sd:.3f}")


    # ---- CLASS-PRESENCE GUARD (anti-collapse) ----
    with torch.no_grad():
        xb, _ = next(iter(dl_val))
        pred = model(xb.to(DEVICE)).argmax(1)
        classes = torch.unique(pred).cpu().tolist()
        print("Classes seen:", classes)

    if 1 not in classes or 2 not in classes:
        print("❌ CLASS COLLAPSE DETECTED. STOPPING.")
        break


    # ---- SAVE BEST (SUP/DEEP mean) ----
    if mean_sd > best_mean:
        best_mean = mean_sd
        torch.save(model.state_dict(), best_path)
        print(f"✅ New best model saved (MEAN(SD)={best_mean:.3f})")


print("\n🏁 Training finished.")
print("Best SUP/DEEP Mean Dice:", best_mean)
print("Model saved to:", best_path)



/tmp/ipython-input-4018571521.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=AMP)
[Train 1]:   0%|          | 0/68 [00:00<?, ?it/s]/tmp/ipython-input-4018571521.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=AMP):
[Train 1]: 100%|██████████| 68/68 [00:36<00:00,  1.88it/s]


[Epoch 01] loss=1.2007 | SUP=0.048  DEEP=0.123  MEAN(SD)=0.085
Classes seen: [0, 1, 2]
✅ New best model saved (MEAN(SD)=0.085)


[Train 2]: 100%|██████████| 68/68 [00:09<00:00,  7.28it/s]


[Epoch 02] loss=0.6724 | SUP=0.087  DEEP=0.279  MEAN(SD)=0.183
Classes seen: [0, 1, 2]
✅ New best model saved (MEAN(SD)=0.183)


[Train 3]: 100%|██████████| 68/68 [00:09<00:00,  7.30it/s]


[Epoch 03] loss=0.2916 | SUP=0.237  DEEP=0.323  MEAN(SD)=0.280
Classes seen: [0, 1, 2]
✅ New best model saved (MEAN(SD)=0.280)


... (70 outputs left out)

[Train 39]: 100%|██████████| 68/68 [00:09<00:00,  7.16it/s]


[Epoch 39] loss=0.0151 | SUP=0.814  DEEP=0.795  MEAN(SD)=0.805
Classes seen: [0, 1, 2]


[Train 40]: 100%|██████████| 68/68 [00:09<00:00,  7.13it/s]


[Epoch 40] loss=0.0147 | SUP=0.823  DEEP=0.800  MEAN(SD)=0.811
Classes seen: [0, 1, 2]

🏁 Training finished.
Best SUP/DEEP Mean Dice: 0.81302210688591
Model saved to: /content/srbd_training_out/best_model.pt


In [ ]:
# === CELL 3a (FIXED): STRICT Dice for SUP / DEEP only ===

import torch, numpy as np

@torch.no_grad()
def evaluate_sup_deep_dice(model, loader, device=None):
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    sup_sum = 0.0
    deep_sum = 0.0
    count = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)

        for cls in [1, 2]:   # SUP=1, DEEP=2
            p = (pred == cls)
            g = (y == cls)
            inter = (p & g).sum().item()
            denom = p.sum().item() + g.sum().item() + 1e-6
            d = 2 * inter / denom

            if cls == 1:
                sup_sum += d
            else:
                deep_sum += d

        count += 1

    sup = sup_sum / count
    deep = deep_sum / count
    mean = (sup + deep) / 2

    print(f"[STRICT] Dice SUP={sup:.3f}  DEEP={deep:.3f}  MEAN={mean:.3f}")
    return sup, deep, mean


# ---- RUN EVALUATION ----
sup, deep, mean = evaluate_sup_deep_dice(model, dl_val)



[STRICT] Dice SUP=0.823  DEEP=0.802  MEAN=0.813


In [ ]:
# === CELL 3b (FIXED): Load Test + Dice + Thickness + Speed ===

!pip install scipy

import pandas as pd
from torch.utils.data import DataLoader
import torch
import numpy as np
import cv2
import time
from pathlib import Path


# ------------------------------------------------
# LOAD TEST SPLIT
# ------------------------------------------------
test_df = pd.read_csv("/content/splits/test.csv")

dl_test = DataLoader(
    SRBDDataset(test_df, tf=val_tf),
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✅ Test samples:", len(dl_test))


# ------------------------------------------------
# LOAD BEST MODEL
# ------------------------------------------------
best_path = "/content/srbd_training_out/best_model.pt"
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint, strict=True)
model.eval()

# SANITY CHECK
with torch.no_grad():
    x,_ = next(iter(dl_val))
    pred = model(x.to(DEVICE)).argmax(1)
    print("✅ Classes after load:", torch.unique(pred).cpu().tolist())


# ------------------------------------------------
# EVALUATION (DICE SUP / DEEP)
# ------------------------------------------------
print("\n✅ VALIDATION PERFORMANCE")
evaluate_sup_deep_dice(model, dl_val)

print("\n✅ TEST PERFORMANCE")
evaluate_sup_deep_dice(model, dl_test)


# =================================================
# AUTO ROI + CORRECT THICKNESS LOGIC
# =================================================

def auto_roi(seg):
    ys, xs = np.where((seg == 1) | (seg == 2))
    if len(xs) == 0 or len(ys) == 0:
        return None
    return xs.min(), ys.min(), xs.max(), ys.max()


def measure_thickness(seg_mask, tag="PRED", min_points=100):

    print(f"\n--- MEASURING {tag} THICKNESS ---")

    roi = auto_roi(seg_mask)
    if roi is None:
        print("❌ No anatomy detected.")
        return None, None

    x1, y1, x2, y2 = roi
    seg = seg_mask[y1:y2, x1:x2]

    sup  = (seg == 1)
    deep = (seg == 2)

    if np.sum(sup) < min_points or np.sum(deep) < min_points:
        print("❌ Too few mask points.")
        return None, None

    thickness = []

    for x in range(seg.shape[1]):
        sup_y  = np.where(sup[:, x])[0]
        deep_y = np.where(deep[:, x])[0]

        if len(sup_y) > 0 and len(deep_y) > 0:
            d = abs(np.median(deep_y) - np.median(sup_y))
            thickness.append(d)

    if len(thickness) == 0:
        print("❌ No valid thickness columns.")
        return None, None

    px = float(np.median(thickness))
    print(f"✅ Thickness = {px:.2f} px")

    return px, None


# =================================================
# FULL-RES INFERENCE + THICKNESS
# =================================================

@torch.no_grad()
def infer_and_measure_fullres(model, img_gray, gt_mask=None):

    model.eval()
    H, W = img_gray.shape

    # Resize for network
    img_rs = cv2.resize(img_gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    x = torch.from_numpy(img_rs[None,None]).float().to(DEVICE) / 255.0

    # Predict
    pred = model(x).argmax(1)[0].cpu().numpy().astype(np.uint8)

    # Restore to original size
    pred = cv2.resize(pred, (W, H), interpolation=cv2.INTER_NEAREST)
    pred[pred > 2] = 0

    pred_px, _ = measure_thickness(pred, tag="PRED")

    if gt_mask is not None:
        gt_px, _ = measure_thickness(gt_mask, tag="GT")
    else:
        gt_px = None

    return pred_px, gt_px


# ------------------------------------------------
# SPEED BENCHMARK
# ------------------------------------------------

def benchmark_speed(model, device="cuda", runs=100, size=(1,1,640,640)):
    model.eval()
    dummy = torch.randn(size).to(device)

    for _ in range(10):
        _ = model(dummy)

    torch.cuda.synchronize()
    t0 = time.time()

    for _ in range(runs):
        _ = model(dummy)

    torch.cuda.synchronize()
    t1 = time.time()

    avg_ms = (t1 - t0) * 1000 / runs
    print(f"\n✅ Inference Speed: {avg_ms:.3f} ms / image")
    return avg_ms

benchmark_speed(model)

print(test_df.columns.tolist())




✅ Test samples: 51
✅ Classes after load: [0, 1, 2]

✅ VALIDATION PERFORMANCE
[STRICT] Dice SUP=0.822  DEEP=0.801  MEAN=0.811

✅ TEST PERFORMANCE
[STRICT] Dice SUP=0.830  DEEP=0.815  MEAN=0.823

✅ Inference Speed: 10.904 ms / image
['image', 'mask_rgb']


In [ ]:
# === CELL 4: FULL-RES INFERENCE + THICKNESS (FINAL FIXED) ===

import numpy as np
import cv2
import torch


# ----------------------------------------------------
# ✅ RGB → CLASS CONVERSION (FOR GROUND TRUTH)
# ----------------------------------------------------
def rgb_to_class_mask(rgb):
    """
    Convert RGB GT → class IDs
    Red    (255,0,0)   → SUP = 1
    Yellow (255,255,0) → DEEP = 2
    Else             → BG = 0
    """

    rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
    mask = np.zeros(rgb.shape[:2], dtype=np.uint8)

    # SUP = Red
    mask[(rgb[:,:,0] == 255) & (rgb[:,:,1] == 0) & (rgb[:,:,2] == 0)] = 1

    # DEEP = Yellow
    mask[(rgb[:,:,0] == 255) & (rgb[:,:,1] == 255) & (rgb[:,:,2] == 0)] = 2

    return mask


# ----------------------------------------------------
# ✅ FULL-RES PREDICTION + MEASUREMENT
# ----------------------------------------------------
@torch.no_grad()
def infer_and_measure_fullres(model, img_gray, gt_mask, device=DEVICE):

    model.eval()

    # ✅ Ensure format
    img_gray = img_gray.astype(np.uint8)
    H, W = img_gray.shape

    # ✅ Resize for network
    img_rs = cv2.resize(img_gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

    # ✅ Torch input
    x = torch.from_numpy(img_rs).unsqueeze(0).unsqueeze(0).float().to(device) / 255.0

    # ✅ Inference
    logits = model(x)
    pred = torch.argmax(logits, dim=1)[0].cpu().numpy().astype(np.uint8)

    # ✅ Resize BACK to original resolution
    pred_full = cv2.resize(pred, (W, H), interpolation=cv2.INTER_NEAREST)

    # ✅ Clamp garbage classes
    pred_full = np.where((pred_full == 1) | (pred_full == 2), pred_full, 0)

    # ✅ Debug
    print("\n--- INFERENCE DEBUG ---")
    print("Pred shape:", pred_full.shape)
    print("Pred unique after resize:", np.unique(pred_full))
    print("SUP pixels:", np.sum(pred_full == 1))
    print("DEEP pixels:", np.sum(pred_full == 2))
    print("Pred sum:", pred_full.sum())

    roi = (0, 0, W, H)

    # ✅ MEASURE PREDICTION
    print("\n--- MEASURING PRED ---")
    pred_px, _ = measure_thickness(pred_full, roi)

    # ✅ MEASURE GT
    print("--- MEASURING GT ---")
    gt_px, _ = measure_thickness(gt_mask, roi)

    return pred_px, gt_px


# ====================================================
# ✅ SINGLE IMAGE SANITY TEST (RUN THIS CELL)
# ====================================================

print("✅ Running single-image sanity test...")

# Locate correct mask column automatically
for col in ["mask", "mask_rgb", "mask_path"]:
    if col in test_df.columns:
        mask_col = col
        break
else:
    raise RuntimeError("❌ No mask column found in test_df")

print("✅ Using mask column:", mask_col)

row = test_df.iloc[0]

# Load input image
img = cv2.imread(row["image"], cv2.IMREAD_GRAYSCALE)

# Load GT mask (RGB)
gt_rgb = cv2.imread(row[mask_col])

# Convert RGB → class mask
gt = rgb_to_class_mask(gt_rgb)

# Run inference + measurement
pred_px, gt_px = infer_and_measure_fullres(model, img, gt)

# Show results
print("\n--- RESULT ---")
print("Pred thickness:", pred_px)
print("GT thickness:", gt_px)





✅ Running single-image sanity test...
✅ Using mask column: mask_rgb

--- INFERENCE DEBUG ---
Pred shape: (436, 528)
Pred unique after resize: [0 1 2]
SUP pixels: 7681
DEEP pixels: 8761
Pred sum: 25203

--- MEASURING PRED ---

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 170.00 px
--- MEASURING GT ---

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 173.50 px

--- RESULT ---
Pred thickness: 170.0
GT thickness: 173.5


In [ ]:
# === CELL 4.5: FULL-RES GT vs PRED THICKNESS MEASUREMENT ===

@torch.no_grad()
def infer_and_measure_fullres(model, img, gt_mask, mm_per_pixel_y=None, device=DEVICE):

    model.eval()

    # resize to training resolution
    img_rs = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    gt_rs  = cv2.resize(gt_mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

    # model inference
    x = torch.from_numpy(img_rs[None, None]).float().to(device) / 255.0
    pred = model(x).argmax(1)[0].cpu().numpy()

    # measure both (IMPORTANT: pred vs GT)
    pred_px, _ = measure_thickness(pred, (0,0,IMG_SIZE,IMG_SIZE), mm_per_pixel_y, tag="PRED")
    gt_px,   _ = measure_thickness(gt_rs, (0,0,IMG_SIZE,IMG_SIZE), mm_per_pixel_y, tag="GT")

    return pred_px, gt_px


In [ ]:
# === CELL 5 (FIXED): REAL THICKNESS ERROR EVALUATION ===

def evaluate_thickness(model, dataloader, mm_per_pixel_y=None):

    errors = []
    skipped = 0

    for img, gt in dataloader:

        img = img[0,0].cpu().numpy()
        gt  = gt[0].cpu().numpy()

        pred_px, gt_px = infer_and_measure_fullres(model, img, gt, mm_per_pixel_y)

        if pred_px is None or gt_px is None:
            skipped += 1
            continue

        errors.append(abs(pred_px - gt_px))

    print("Skipped samples:", skipped)

    if len(errors) == 0:
        return None, None

    return float(np.mean(errors)), float(np.median(errors))




In [ ]:
@torch.no_grad()
def infer_and_measure_fullres(model, img_gray, gt_mask, device=DEVICE):

    model.eval()
    H, W = img_gray.shape

    # Resize FOR inference
    img_rs = cv2.resize(img_gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    x = torch.from_numpy(img_rs[None,None]).float().to(device)/255.0

    # Predict
    pred = model(x).argmax(1)[0].cpu().numpy().astype(np.uint8)

    # Resize BACK to original
    pred_full = cv2.resize(pred, (W,H), interpolation=cv2.INTER_NEAREST)

    pred_full[pred_full > 2] = 0

    roi = (0, 0, W, H)

    # Measure both FULL-RES
    pred_px, _ = measure_thickness(pred_full, roi)
    gt_px,   _ = measure_thickness(gt_mask, roi)

    return pred_px, gt_px



In [ ]:
# === CELL 6: GT vs PRED THICKNESS ERROR (FINAL FIX) ===

errors = []
skipped = 0

print("✅ Evaluating thickness on test set...")

for i, row in test_df.iterrows():

    img = cv2.imread(row["image"], cv2.IMREAD_GRAYSCALE)
    gt_rgb = cv2.imread(row["mask_rgb"])
    gt = rgb_to_class_mask(gt_rgb)

    pred_px, gt_px = infer_and_measure_fullres(model, img, gt)

    print(f"\nSample {i}")
    print("Pred thickness:", pred_px)
    print("GT thickness:", gt_px)

    if pred_px is None or gt_px is None:
        print("🚫 SKIPPED")
        skipped += 1
        continue

    errors.append(abs(pred_px - gt_px))


print("\n✅ TOTAL:", len(test_df))
print("🚫 SKIPPED:", skipped)

if len(errors):
    print("✅ MEAN ERROR (px):", np.mean(errors))
    print("✅ MEDIAN ERROR (px):", np.median(errors))
    print("✅ % < 3px:", np.mean(np.array(errors) < 3) * 100)
    print("✅ % < 5px:", np.mean(np.array(errors) < 5) * 100)
else:
    print("❌ No valid samples")





✅ Evaluating thickness on test set...

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 170.00 px

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 173.50 px

Sample 0
Pred thickness: 170.0
GT thickness: 173.5

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 246.50 px

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 250.00 px

Sample 1
Pred thickness: 246.5
GT thickness: 250.0

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 174.00 px

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 177.00 px

Sample 2
Pred thickness: 174.0
GT thickness: 177.0

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 176.50 px

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 182.00 px

Sample 3
Pred thickness: 176.5
GT thickness: 182.0

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 182.00 px

--- MEASURING (0, 0, 528, 436) THICKNESS ---
✅ Thickness = 183.00 px

Sample 4
Pred thickness: 182.0
GT thickness: 183.0

-

In [ ]:
# === EXPORT MODEL (RUN AS SEPARATE CELL) ===
!pip install onnx onnxscript onnxruntime

dummy = torch.randn(1,1,IMG_SIZE,IMG_SIZE).to(DEVICE)

onnx_path = "/content/srbd_training_out/apo_model.onnx"
torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=11
)

print("✅ ONNX exported:", onnx_path)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.0/683.0 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.3 MB/s eta 0:00:00


/tmp/ipython-input-4181909989.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1204 11:11:52.441000 2808 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Unet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Unet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:65: adapter_lookup: Assertion `false`

Applied 117 of general pattern rewrite rules.
✅ ONNX exported: /content/srbd_training_out/apo_model.onnx


In [ ]:
# === WPF INFERENCE FUNCTIONS (FULL FIXED CELL) ===

import json
import os

def predict_thickness_for_ui(image_path):

    # --- Load image safely ---
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        print("❌ ERROR: Image not found or unreadable ->", image_path)
        return None

    H, W = img.shape

    # --- Resize for model ---
    img_rs = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    x = torch.from_numpy(img_rs[None,None]).float().to(DEVICE) / 255.0

    # --- Inference ---
    with torch.no_grad():
        pred = model(x).argmax(1)[0].cpu().numpy().astype(np.uint8)

    # --- Restore to original resolution ---
    pred = cv2.resize(pred, (W, H), interpolation=cv2.INTER_NEAREST)
    pred[pred > 2] = 0

    # --- Thickness measurement ---
    thickness_px, _ = measure_thickness(pred, tag="PRED")

    # --- Save mask for UI overlay ---
    overlay = (pred * 80).astype(np.uint8)
    cv2.imwrite("seg_result.png", overlay)

    print("✅ Segmentation saved as seg_result.png")

    return thickness_px


def run_for_wpf(image_path, out_json="result.json"):

    thickness = predict_thickness_for_ui(image_path)

    result = {
        "image": os.path.basename(image_path),
        "thickness_px": None if thickness is None else round(float(thickness), 2),
        "status": "ok" if thickness is not None else "failed"
    }

    with open(out_json, "w") as f:
        json.dump(result, f, indent=2)

    print("✅ Output JSON created:", out_json)



In [ ]:
# === CELL C: TEST WPF PIPELINE (FINAL FIXED VERSION) ===

import os

# List all available images
path = "/content/RGB_READY/RyanCunningham/images"
files = os.listdir(path)

print("✅ Available images:")
for f in files[:20]:   # show first 20 only
    print(" -", f)

# Pick the first valid image automatically
if len(files) == 0:
    print("❌ No images found in folder.")
else:
    test_img = os.path.join(path, files[0])
    print("\n📌 Testing with:", test_img)

    run_for_wpf(test_img)



✅ Available images:
 - 39.jpg
 - 284.jpg
 - 442.jpg
 - 249.jpg
 - 313.jpg
 - 265.jpg
 - 272.jpg
 - 333.jpg
 - 101.jpg
 - 193.jpg
 - 264.jpg
 - 437.jpg
 - 158.jpg
 - 293.jpg
 - 58.jpg
 - 96.jpg
 - 369.jpg
 - 204.jpg
 - 415.jpg
 - 394.jpg

📌 Testing with: /content/RGB_READY/RyanCunningham/images/39.jpg

--- MEASURING PRED THICKNESS ---
✅ Thickness = 195.00 px
✅ Segmentation saved as seg_result.png
✅ Output JSON created: result.json


In [ ]:
import os
os.path.exists("result.json"), os.path.exists("seg_result.png")


(True, True)